## Weather Agent with Google ADK

This notebook demonstrates how to build an AI agent that retrieves
real-time weather forecasts using external APIs.

Tools used:
• Google Maps Geocoding API
• National Weather Service API

The agent converts city names into coordinates and retrieves
weather forecasts for multiple US cities. Two agents are tested:

• Gemini (gemini-2.0-flash)
• GPT-4.1 via LiteLLM

## 1. Setup & Dependencies

In [1]:
!pip install google-adk -q
!pip install litellm -q

In [20]:
import os
import requests
from typing import Tuple, Dict
from google.genai import types
import google.auth
import time

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner

## 2. Configuration

Set your API keys here. The Google Maps key is required for geocoding.
The OpenAI key is needed for the GPT-based agent.

In [21]:
project = google.auth.default()[1]

# Set API keys — replace with your own or set as environment variables
GOOGLE_MAPS_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY", "YOUR_KEY_HERE")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_KEY_HERE")

# LiteLlm reads this from the environment for OpenAI calls
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = project
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

## 3. Tool Functions

In [4]:
def geocode_location(place: str, api_key: str) -> Tuple[float, float]:
    """Convert a place name into latitude and longitude.

    Uses the Google Maps Geocoding API to resolve a human-readable
    location string into geographic coordinates.

    Args:
        place: Name of the location to geocode (e.g., "Chicago, IL").
        api_key: Google Maps Geocoding API key.

    Returns:
        A tuple of (latitude, longitude) as floats.

    Raises:
        ValueError: If the geocoding API returns no results for the place.
        requests.RequestException: If the API request fails.
    """
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": place, "key": api_key}

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    if not data.get("results"):
        raise ValueError(f"No geocoding results found for '{place}'.")

    location = data["results"][0]["geometry"]["location"]
    return location["lat"], location["lng"]

In [5]:
def get_weather_forecast(latitude: float, longitude: float) -> Dict:
    """Retrieve the current weather forecast from the National Weather Service.

    Makes two sequential calls to the NWS API: first to resolve the
    coordinates to a forecast office grid, then to fetch the forecast.

    Args:
        latitude: Latitude of the location.
        longitude: Longitude of the location.

    Returns:
        A dictionary representing the first (current) forecast period,
        including keys like 'name', 'temperature', 'detailedForecast', etc.

    Raises:
        KeyError: If the API response structure is unexpected.
        requests.RequestException: If either API request fails.
    """
    points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
    points_response = requests.get(points_url)
    points_response.raise_for_status()
    points_data = points_response.json()

    forecast_url = points_data["properties"]["forecast"]
    forecast_response = requests.get(forecast_url)
    forecast_response.raise_for_status()
    forecast_data = forecast_response.json()

    return forecast_data["properties"]["periods"][0]

In [33]:
def weather_lookup(city: str) -> str:
    """Retrieve a weather summary for a US city.

    Args:
        city: A US city name, ideally with state (e.g., "Denver, CO").

    Returns:
        A string containing the detailed weather forecast.
    """
    try:
        time.sleep(2)  # Avoid rate limiting on the Geocoding API
        lat, lon = geocode_location(city, GOOGLE_MAPS_API_KEY)
        forecast = get_weather_forecast(lat, lon)
        return forecast["detailedForecast"]
    except (ValueError, KeyError, requests.RequestException) as e:
        return f"Unable to retrieve weather for '{city}': {e}"

## 4. Agent Definitions

Two agents with the same tools and purpose, backed by different models.

In [7]:
AGENT_INSTRUCTION = """
You are a helpful weather assistant for US cities.

When a user asks about the weather in a city, use the weather_lookup
tool to retrieve the current forecast. Then provide:
  1. A clear, concise summary of current conditions.
  2. The temperature and any notable weather alerts or advisories.
  3. A brief recommendation (e.g., bring an umbrella, wear layers).

If the tool returns an error, let the user know and suggest they
check the city name or try again.
"""

# Gemini-backed agent
weather_agent_gemini = Agent(
    name="weather_agent_gemini",
    model="gemini-2.0-flash",
    description="Provides weather forecasts for US cities using Gemini.",
    instruction=AGENT_INSTRUCTION,
    tools=[weather_lookup],
)

# GPT-backed agent (via LiteLlm)
weather_agent_gpt = Agent(
    name="weather_agent_gpt",
    model=LiteLlm(model="openai/gpt-4.1"),
    description="Provides weather forecasts for US cities using GPT.",
    instruction=AGENT_INSTRUCTION,
    tools=[weather_lookup],
)

## 5. Test Harness

Run both agents against multiple US cities to demonstrate functionality.

In [22]:


async def test_agent(agent: Agent, cities: list[str]) -> None:
    """Run a weather agent against a list of cities and print results."""
    session_service = InMemorySessionService()
    runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)

    print(f"\n{'=' * 60}")
    print(f"Testing agent: {agent.name}")
    print(f"{'=' * 60}")

    for i, city in enumerate(cities):
      session_id = f"test_session_{i}"

      # Create the session first
      await session_service.create_session(
          app_name=agent.name,
          user_id="test_user",
          session_id=session_id,
      )

      content = types.Content(
          role="user",
          parts=[types.Part(text=f"What is the weather in {city}?")]
      )

      async for event in runner.run_async(
          user_id="test_user",
          session_id=session_id,
          new_message=content,
      ):
          if event.is_final_response():
              result = event.content.parts[0].text
              print(f"\n--- {city} ---")
              print(result)

In [9]:
TEST_CITIES = [
    "Chicago, IL",
    "Denver, CO",
    "Miami, FL",
    "Seattle, WA",
    "New York, NY",
]

### 5a. Test with Gemini

In [31]:
await test_agent(weather_agent_gemini, TEST_CITIES)


Testing agent: weather_agent_gemini

--- Chicago, IL ---
The weather in Chicago, IL is cloudy with widespread fog and a chance of rain showers before noon, then areas of fog and a slight chance of drizzle. The high will be near 39°F. There is a 30% chance of precipitation, with new rainfall amounts less than a tenth of an inch possible. Consider taking an umbrella.

--- Denver, CO ---
The weather in Denver, CO is mostly sunny with a high near 68, falling to around 65 in the afternoon. Expect a south southwest wind of 5 to 10 mph, with gusts as high as 20 mph. It would be a good idea to wear layers.


--- Miami, FL ---
The weather in Miami, FL is mostly sunny with a slight chance of showers and thunderstorms after 5pm. The high will be near 79 degrees. Expect an east wind around 15 mph, with gusts as high as 21 mph. The chance of precipitation is 20%.

--- Seattle, WA ---
The weather in Seattle is mostly cloudy with a chance of rain before 4pm. The high will be near 50 degrees. There i

### 5b. Test with GPT

In [34]:
await test_agent(weather_agent_gpt, TEST_CITIES)


Testing agent: weather_agent_gpt

--- Chicago, IL ---
Current weather in Chicago, IL:
- There is widespread fog and a chance of rain showers before noon, followed by areas of fog and a slight chance of drizzle. It will remain cloudy throughout the day.
- The high temperature is around 39°F with a light north wind around 5 mph. Chance of precipitation is 30%. No significant weather alerts are reported.
- Recommendation: Visibility may be reduced due to fog, so drive carefully. You might want a light jacket and an umbrella just in case of drizzle.

--- Denver, CO ---
Currently in Denver, CO, it is mostly sunny with a high near 68°F, cooling to around 65°F in the afternoon. Winds are coming from the south-southwest at 5 to 10 mph, with gusts up to 20 mph. There are no notable weather alerts or advisories at this time.

You may want to wear light layers—a jacket might be useful if you’ll be out when temperatures cool in the afternoon. Enjoy the pleasant weather!

--- Miami, FL ---
Current